<a href="https://colab.research.google.com/github/sehir-shah/Hospital-AI-chatbot/blob/main/hospital-ai-chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!python -m spacy download en_core_web_md

In [ ]:
!pip install gradio

In [ ]:
import spacy
import gradio as gr

# Load the AI model
try:
    nlp = spacy.load("en_core_web_md")
    print("🚀 Perfect AI Model Loaded!")
except OSError:
    print("❌ Error: Run Cell 1 first.")

# Upgraded Knowledge Base
HOSPITAL_KNOWLEDGE = [
    {
        "intent": "appointments",
        "anchors": ["book", "appointment", "schedule", "slot", "registration"],
        "phrases": ["book an appointment", "schedule doctor visit", "make appointment", "see a doctor", "book a slot"],
        "response": "🗓️ **Appointment Booking:**\n• **Online:** www.centralhospital.local/bookings\n• **Phone:** (555) 0199 (8 AM - 6 PM)."
    },
    {
        "intent": "timings",
        "anchors": ["timing", "hours", "open", "close", "opd", "saturday", "sunday", "weekday", "lab"],
        "phrases": ["hospital timings", "when do you open", "opd hours", "working hours", "is it open"],
        "response": "🕒 **Hospital Timings:**\n• **OPD:** Mon - Sat, 9:00 AM – 5:00 PM.\n• **ER & Emergency:** Open 24/7.\n• **Laboratory:** Mon - Sat, 7:00 AM – 7:00 PM."
    },
    {
        "intent": "fees",
        "anchors": ["fee", "fees", "charge", "charges", "cost", "price", "how much", "payment"],
        "phrases": ["consultation fee", "doctor charges", "cost of checkup", "price", "visiting fees", "how much do you charge"],
        "response": "💰 **Fee Structure:**\n• General Physician: $50\n• Specialist (Cardiology/Dermatology): $90 - $120\n• Emergency ER Base: $150"
    },
    {
        "intent": "emergency",
        "anchors": ["emergency", "ambulance", "er", "911", "accident", "chest pain", "urgent"],
        "phrases": ["emergency number", "ambulance contact", "urgent help", "chest pain", "accident"],
        "response": "🚨 **EMERGENCY:**\n• Immediate Crisis: Call **911**\n• ER Hotline: (555) 0100\n• Ambulance: (555) 0111"
    },
    {
        "intent": "visiting_hours",
        "anchors": ["visit", "visiting", "visitor", "visitors", "icu", "ward", "patient visit", "guest"],
        "phrases": ["visiting hours", "can I visit a patient", "when can family visit", "icu visiting hours"],
        "response": "👥 **Patient Visiting Hours:**\n• **General Wards:** Daily from 10:00 AM – 12:00 PM and 4:00 PM – 8:00 PM.\n• **Intensive Care Unit (ICU):** Strictly 5:00 PM – 6:00 PM (Immediate family only)."
    }
]

def chatbot_response(message, history):
    clean_input = str(message).strip().lower()

    # ----------------------------------------------------
    # PRIORITY 1: Specific Sub-Intent Matching (STRICT SPECIFICS ALWAYS WIN FIRST)
    # ----------------------------------------------------
    if "skin" in clean_input or "ski" in clean_input or "derm" in clean_input:
        return "🩸 **Dermatology Department (Skin Specialist):**\nOur skin specialists handle allergies, treatments, and cosmetic concerns.\n• **OPD Timings:** Monday to Friday, 10:00 AM – 4:00 PM.\n• *Note: Appointments are required.*"

    if "heart" in clean_input or "cardio" in clean_input:
        return "❤️ **Cardiology Department:**\nOur heart specialists are available in the OPD Monday through Saturday, 9:00 AM – 5:00 PM. For acute chest pain or cardiac distress, please go directly to our 24/7 ER."

    if "child" in clean_input or "kid" in clean_input or "pediat" in clean_input or "baby" in clean_input:
        return "👶 **Pediatrics Department:**\nWe provide complete newborn checkups, general childcare, and scheduled vaccinations. Open Monday to Saturday, 9:00 AM – 5:00 PM."

    if "bone" in clean_input or "fractur" in clean_input or "ortho" in clean_input or "joint" in clean_input:
        return "🦴 **Orthopedics Department:**\nSpecialists in bone fractures, joint pain, and physical therapy rehabilitation. Open Monday to Saturday, 9:00 AM – 5:00 PM."

    # ----------------------------------------------------
    # PRIORITY 2: General Department Overviews (Only triggers if NO specific doctor was named)
    # ----------------------------------------------------
    if "department" in clean_input or "specialities" in clean_input or "speciality" in clean_input or "doctor list" in clean_input:
        return "🏥 **Our Specialised Departments:**\n• **Cardiology** (Heart care)\n• **Dermatology** (Skin care)\n• **Pediatrics** (Child healthcare)\n• **Orthopedics** (Bone & Joint)"

    # ----------------------------------------------------
    # PRIORITY 3: General Knowledge Base Anchors (Appointments, Fees, Timings)
    # ----------------------------------------------------
    for category in HOSPITAL_KNOWLEDGE:
        for anchor in category["anchors"]:
            if anchor in clean_input:
                return category["response"]

    # ----------------------------------------------------
    # PRIORITY 4: Semantic NLP Similarity (Fallback)
    # ----------------------------------------------------
    user_doc = nlp(clean_input)
    best_match = None
    highest_score = 0.0

    for category in HOSPITAL_KNOWLEDGE:
        for phrase in category["phrases"]:
            phrase_doc = nlp(phrase.lower())
            score = user_doc.similarity(phrase_doc)
            if score > highest_score:
                highest_score = score
                best_match = category

    if highest_score >= 0.45 and best_match:
        return best_match["response"]
    else:
        return (
            "🤖 I'm not entirely sure. I can help you with:\n"
            "• Booking appointments\n"
            "• Hospital timings & OPD hours\n"
            "• Department info (Heart, Skin, Child care...)\n"
            "• Consultation fees & Emergency numbers\n\n"
            "Could you try rephrasing your question?"
        )

# Launch the corrected GUI
demo = gr.ChatInterface(
    fn=chatbot_response,
    title="🏥 Central Hospital AI Assistant",
    description="Ask me about hospital timings, specific departments, fees, or booking details.",
    theme="soft"
)

demo.launch(inline=True, share=False)